# INV-M365-CB — Authoritative Persona Certification Count Rebase v1

**Purpose:** Derive the staged H4 certification and count truth directly from the authoritative persona registry and rebased department-pack authority.

**Lemma mapping:** `docs/ma/lemmas/L80_m365_authoritative_persona_certification_count_rebase_v1.md`

**Invariant support:** `invariants/lemmas/L80_m365_authoritative_persona_certification_count_rebase_v1.yaml`

**Expected results:** Deterministically compute the H4 staged truth of `59 total / 10 departments / 34 active / 25 planned`, derive persona and department certification KPI targets, record the stale pre-H4 certification/count claims, and write `configs/generated/authoritative_persona_certification_count_rebase_v1_verification.json`.

This notebook is Phase 5 notebook development for H4. It reads only the current authoritative registry, rebased department packs, and the scoped stale certification/count surfaces. It does not mutate those certification surfaces; it computes the staged truth that later extraction must mirror exactly.

In [1]:
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

import yaml

repo = Path.cwd().resolve()
while not (repo / 'registry' / 'persona_registry_v2.yaml').exists():
    if repo.parent == repo:
        raise RuntimeError('repo root not found')
    repo = repo.parent

scoped_paths = [
    repo / 'registry' / 'persona_certification_v1.yaml',
    repo / 'registry' / 'department_certification_v1.yaml',
    repo / 'registry' / 'enterprise_release_gate_v2.yaml',
    repo / 'docs' / 'commercialization' / 'm365-persona-certification-v1.md',
    repo / 'docs' / 'commercialization' / 'm365-department-certification-v1.md',
    repo / 'docs' / 'commercialization' / 'm365-persona-registry-v2.md',
    repo / 'docs' / 'commercialization' / 'm365-department-persona-census.md',
    repo / 'docs' / 'commercialization' / 'm365-enterprise-release-gate-v2.md',
]

registry = yaml.safe_load((repo / 'registry' / 'persona_registry_v2.yaml').read_text(encoding='utf-8'))
personas = registry['personas']
summary = registry['summary']

assert summary['total_personas'] == 59
assert summary['total_departments'] == 10
assert summary['active_personas'] == 34
assert summary['planned_personas'] == 25
assert summary['registry_backed_personas'] == 34
assert summary['persona_contract_only_personas'] == 25

summary

{'total_personas': 59,
 'total_departments': 10,
 'active_personas': 34,
 'planned_personas': 25,
 'registry_backed_personas': 34,
 'persona_contract_only_personas': 25}

Next, compute the exact staged persona, risk, and department totals from the registry and rebased department packs. These are the only values H4 is allowed to claim.

In [2]:
risk_counts: dict[str, int] = {}
status_counts: dict[str, int] = {}
coverage_counts: dict[str, int] = {}
dept_counts: dict[str, dict[str, int | str]] = {}
total_routed_actions = 0

for persona in personas.values():
    risk = persona['risk_tier']
    risk_counts[risk] = risk_counts.get(risk, 0) + 1

    status = persona['status']
    status_counts[status] = status_counts.get(status, 0) + 1

    coverage = persona['coverage_status']
    coverage_counts[coverage] = coverage_counts.get(coverage, 0) + 1

    dept = dept_counts.setdefault(
        persona['department'],
        {
            'persona_count': 0,
            'active_persona_count': 0,
            'planned_persona_count': 0,
            'registry_backed_persona_count': 0,
            'contract_only_persona_count': 0,
            'workflow_family_count': 0,
            'workload_family_count': 0,
            'department_status': '',
        },
    )
    dept['persona_count'] += 1
    if status == 'active':
        dept['active_persona_count'] += 1
    elif status == 'planned':
        dept['planned_persona_count'] += 1
    if coverage == 'registry-backed':
        dept['registry_backed_persona_count'] += 1
    elif coverage == 'persona-contract-only':
        dept['contract_only_persona_count'] += 1
    total_routed_actions += int(persona['action_count'])

for pack_path in sorted((repo / 'registry').glob('department_pack_*_v1.yaml')):
    pack = yaml.safe_load(pack_path.read_text(encoding='utf-8'))
    dept_id = pack['department']['id']
    dept = dept_counts[dept_id]
    dept['workflow_family_count'] = len(pack.get('workflow_families', []))
    dept['workload_family_count'] = len(pack.get('workload_families', []))
    dept['department_status'] = pack['department']['status']
    assert len(pack.get('personas', {})) == dept['persona_count']
    assert pack['kpis']['required_personas'] == dept['persona_count']
    assert pack['kpis']['required_active_personas'] == dept['active_persona_count']
    assert pack['kpis']['required_registry_backed_personas'] == dept['registry_backed_persona_count']
    assert pack['kpis']['required_workflow_families'] == dept['workflow_family_count']

assert risk_counts == {'critical': 5, 'high': 8, 'medium': 14, 'low': 32}
assert status_counts == {'active': 34, 'planned': 25}
assert coverage_counts == {'registry-backed': 34, 'persona-contract-only': 25}
assert total_routed_actions == 298

dept_counts

{'engineering': {'persona_count': 8,
  'active_persona_count': 7,
  'planned_persona_count': 1,
  'registry_backed_persona_count': 7,
  'contract_only_persona_count': 1,
  'workflow_family_count': 4,
  'workload_family_count': 10,
  'department_status': 'partial-activation'},
 'studio-operations': {'persona_count': 9,
  'active_persona_count': 5,
  'planned_persona_count': 4,
  'registry_backed_persona_count': 5,
  'contract_only_persona_count': 4,
  'workflow_family_count': 5,
  'workload_family_count': 8,
  'department_status': 'partial-activation'},
 'testing': {'persona_count': 5,
  'active_persona_count': 5,
  'planned_persona_count': 0,
  'registry_backed_persona_count': 5,
  'contract_only_persona_count': 0,
  'workflow_family_count': 4,
  'workload_family_count': 4,
  'department_status': 'registry-backed'},
 'marketing': {'persona_count': 8,
  'active_persona_count': 2,
  'planned_persona_count': 6,
  'registry_backed_persona_count': 2,
  'contract_only_persona_count': 6,
  'w

Record the frozen H4 preflight stale-claim inventory captured before extraction. This keeps the proof replay deterministic after the scoped certification surfaces are updated.

In [3]:
preflight_stale_claims = [
    {
        "line": 120,
        "match": "39 personas",
        "path": "registry/persona_certification_v1.yaml"
    },
    {
        "line": 126,
        "match": "39 personas",
        "path": "registry/persona_certification_v1.yaml"
    },
    {
        "line": 121,
        "match": "4 registry-backed personas",
        "path": "registry/persona_certification_v1.yaml"
    },
    {
        "line": 122,
        "match": "35 contract-only personas",
        "path": "registry/persona_certification_v1.yaml"
    },
    {
        "line": 101,
        "match": "total_personas: 39",
        "path": "registry/persona_certification_v1.yaml"
    },
    {
        "line": 140,
        "match": "39 total personas",
        "path": "registry/department_certification_v1.yaml"
    },
    {
        "line": 91,
        "match": "total_department_personas: 39",
        "path": "registry/department_certification_v1.yaml"
    },
    {
        "line": 34,
        "match": "39 personas",
        "path": "registry/enterprise_release_gate_v2.yaml"
    },
    {
        "line": 37,
        "match": "39 personas",
        "path": "registry/enterprise_release_gate_v2.yaml"
    },
    {
        "line": 110,
        "match": "39 personas",
        "path": "registry/enterprise_release_gate_v2.yaml"
    },
    {
        "line": 111,
        "match": "39 personas",
        "path": "registry/enterprise_release_gate_v2.yaml"
    },
    {
        "line": 116,
        "match": "39 personas",
        "path": "registry/enterprise_release_gate_v2.yaml"
    },
    {
        "line": 130,
        "match": "39 personas",
        "path": "registry/enterprise_release_gate_v2.yaml"
    },
    {
        "line": 116,
        "match": "35 of 39 personas",
        "path": "registry/enterprise_release_gate_v2.yaml"
    },
    {
        "line": 102,
        "match": "personas_certified: 39",
        "path": "registry/enterprise_release_gate_v2.yaml"
    },
    {
        "line": 39,
        "match": "39 personas",
        "path": "docs/commercialization/m365-persona-certification-v1.md"
    },
    {
        "line": 40,
        "match": "4 registry-backed personas",
        "path": "docs/commercialization/m365-persona-certification-v1.md"
    },
    {
        "line": 41,
        "match": "35 contract-only personas",
        "path": "docs/commercialization/m365-persona-certification-v1.md"
    },
    {
        "line": 30,
        "match": "39 total personas",
        "path": "docs/commercialization/m365-department-certification-v1.md"
    },
    {
        "line": 16,
        "match": "39 personas",
        "path": "docs/commercialization/m365-enterprise-release-gate-v2.md"
    },
    {
        "line": 29,
        "match": "39 personas",
        "path": "docs/commercialization/m365-enterprise-release-gate-v2.md"
    },
    {
        "line": 29,
        "match": "35 of 39 personas",
        "path": "docs/commercialization/m365-enterprise-release-gate-v2.md"
    }
]

assert len(preflight_stale_claims) == 22
preflight_stale_claims[:12]


[{'line': 120,
  'match': '39 personas',
  'path': 'registry/persona_certification_v1.yaml'},
 {'line': 126,
  'match': '39 personas',
  'path': 'registry/persona_certification_v1.yaml'},
 {'line': 121,
  'match': '4 registry-backed personas',
  'path': 'registry/persona_certification_v1.yaml'},
 {'line': 122,
  'match': '35 contract-only personas',
  'path': 'registry/persona_certification_v1.yaml'},
 {'line': 101,
  'match': 'total_personas: 39',
  'path': 'registry/persona_certification_v1.yaml'},
 {'line': 140,
  'match': '39 total personas',
  'path': 'registry/department_certification_v1.yaml'},
 {'line': 91,
  'match': 'total_department_personas: 39',
  'path': 'registry/department_certification_v1.yaml'},
 {'line': 34,
  'match': '39 personas',
  'path': 'registry/enterprise_release_gate_v2.yaml'},
 {'line': 37,
  'match': '39 personas',
  'path': 'registry/enterprise_release_gate_v2.yaml'},
 {'line': 110,
  'match': '39 personas',
  'path': 'registry/enterprise_release_gate_v2

Build the deterministic H4 verification payload. This is the artifact the extracted certification/count surfaces must satisfy exactly.

In [4]:
verification = {
    'lemma_id': 'L80',
    'determinism_inputs': {
        'registry': 'registry/persona_registry_v2.yaml',
        'department_pack_glob': 'registry/department_pack_*_v1.yaml',
        'scoped_surfaces': [p.relative_to(repo).as_posix() for p in scoped_paths],
    },
    'staged_summary': {
        'total_personas': summary['total_personas'],
        'total_departments': summary['total_departments'],
        'active_personas': summary['active_personas'],
        'planned_personas': summary['planned_personas'],
        'registry_backed_personas': summary['registry_backed_personas'],
        'persona_contract_only_personas': summary['persona_contract_only_personas'],
        'total_routed_actions': total_routed_actions,
    },
    'risk_distribution': risk_counts,
    'status_distribution': status_counts,
    'coverage_distribution': coverage_counts,
    'department_certification_targets': dict(sorted(dept_counts.items())),
    'preflight_stale_claim_count': len(preflight_stale_claims),
    'preflight_stale_claims': preflight_stale_claims,
    'anti_overclaim_rules': {
        'max_active_personas_before_h5': 34,
        'max_registry_backed_personas_before_h5': 34,
        'forbidden_claims': [
            'all 59 personas have implemented actions',
            'all 59 personas are action-capable',
            'all 59 personas are active before H5',
        ],
    },
}

canonical = json.dumps(verification, indent=2, sort_keys=True)
verification['deterministic_hash'] = hashlib.sha256(canonical.encode('utf-8')).hexdigest()

output_path = repo / 'configs' / 'generated' / 'authoritative_persona_certification_count_rebase_v1_verification.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(verification, indent=2, sort_keys=True) + '\n', encoding='utf-8')

roundtrip = json.loads(output_path.read_text(encoding='utf-8'))
assert roundtrip['staged_summary']['total_personas'] == 59
assert roundtrip['staged_summary']['active_personas'] == 34
assert roundtrip['staged_summary']['planned_personas'] == 25
assert roundtrip['staged_summary']['total_routed_actions'] == 298

{'output_path': output_path.as_posix(), 'deterministic_hash': verification['deterministic_hash']}

{'output_path': '/Users/smarthaus/Projects/GitHub/M365/configs/generated/authoritative_persona_certification_count_rebase_v1_verification.json',
 'deterministic_hash': '6673171d5f9645addcd07f5fade8b7917eca4130c589e60be10299ee5e35f78e'}